# Linear Discriminant Analysis (LDA) 

In [1]:
# Standard imports 
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

# Dataset
from sklearn.datasets import load_wine

# For modeling 
from sklearn import linear_model, datasets
import itertools

# For graphing
import matplotlib.colors as colors
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits import mplot3d

# For Evaluation
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [2]:
def multivariate_gaussian_pdf(X, MU, SIGMA):
    """
    Calculate the probability density of a multivariate Gaussian distribution.

    Parameters:
    X (ndarray): A numpy array representing the point at which the probability density is evaluated.
    MU (ndarray): A numpy array representing the mean vector of the Gaussian distribution.
    SIGMA (ndarray): A numpy array representing the covariance matrix of the Gaussian distribution.

    Returns:
    float: The probability density of the multivariate Gaussian distribution at point X.
    """

    # Reshape X and MU to ensure they are column vectors.
    X = X.reshape(-1, 1)
    MU = MU.reshape(-1, 1)

    # Get the dimension of the Gaussian distribution.
    p, _ = SIGMA.shape

    # Compute the inverse of the covariance matrix.
    SIGMA_inv = np.linalg.inv(SIGMA)

    # Calculate the Gaussian probability density function.
    denominator = np.sqrt((2 * np.pi)**p * np.linalg.det(SIGMA))
    exponent = -(1/2) * ((X - MU).T @ SIGMA_inv @ (X - MU))

    # Extract the scalar value from the 1x1 matrix to avoid deprecation warning
    exponent_scalar = exponent[0, 0]
    
    return float((1. / denominator) * np.exp(exponent_scalar))


def calculate_boundary(X, MU_k, MU_l, SIGMA, pi_k, pi_l):
    """
    Calculate the decision boundary for a binary classifier based on the Gaussian distribution.

    Parameters:
    X (ndarray): A numpy array representing the data point at which the decision boundary is evaluated.
    MU_k (ndarray): The mean vector of the Gaussian distribution for class k.
    MU_l (ndarray): The mean vector of the Gaussian distribution for class l.
    SIGMA (ndarray): The shared covariance matrix for the Gaussian distributions.
    pi_k (float): The prior probability of class k.
    pi_l (float): The prior probability of class l.

    Returns:
    float: The value of the decision boundary function at point X.
    """

    # Calculate the inverse of the covariance matrix.
    SIGMA_inv = np.linalg.inv(SIGMA)

    # Compute the terms for the log of the ratio of the prior probabilities.
    log_prior_ratio = np.log(pi_k / pi_l)
    means_term = -1/2 * (MU_k + MU_l).T @ SIGMA_inv @ (MU_k - MU_l)
    x_term = X.T @ SIGMA_inv @ (MU_k - MU_l)

    # Sum the terms and flatten the result to get a scalar value.
    decision_value = (log_prior_ratio + means_term + x_term).flatten()[0]

    return decision_value


def LDA_score(X, MU_k, SIGMA, pi_k):
    """
    Compute the Linear Discriminant Analysis (LDA) score for a specific class and data point.

    Parameters:
    X (ndarray): A numpy array representing the data point for which the LDA score is calculated.
    MU_k (ndarray): The mean vector for class k.
    SIGMA (ndarray): The covariance matrix, assumed to be the same for all classes in LDA.
    pi_k (float): The prior probability of class k.

    Returns:
    float: The LDA score for class k given the data point X.
    """

    # Compute the inverse of the covariance matrix
    SIGMA_inv = np.linalg.inv(SIGMA)

    # Log of the prior probability for class k
    log_prior = np.log(pi_k)

    # The LDA score
    quadratic_term = -1/2 * (MU_k).T @ SIGMA_inv @ MU_k
    linear_term = X.T @ SIGMA_inv @ MU_k
    lda_score = (log_prior + quadratic_term + linear_term).flatten()[0]

    return lda_score


def predict_LDA_class(X, MU_list, SIGMA, pi_list):
    """
    Predict the class label for a data point using Linear Discriminant Analysis (LDA).

    Parameters:
    X (ndarray): A numpy array representing the data point to be classified.
    MU_list (list of ndarray): A list of mean vectors, one for each class.
    SIGMA (ndarray): The shared covariance matrix used in LDA.
    pi_list (list): A list of prior probabilities, one for each class.

    Returns:
    int: The predicted class label for the data point X, identified by the highest LDA score.
    """

    scores_list = []
    classes = len(MU_list)

    # Compute the LDA score for each class and append it to the scores list
    for p in range(classes):
        score = LDA_score(X.reshape(-1,1), MU_list[p].reshape(-1,1), SIGMA, pi_list[p])
        scores_list.append(score)

    # Return the index of the class with the highest score
    # This index corresponds to the predicted class label
    return np.argmax(scores_list)

The wine dataset in scikit-learn is a classic dataset commonly used in machine learning for classification tasks. This dataset is included in the sklearn.datasets module and can be easily loaded for experimentation and educational purposes. Here's a breakdown of its key characteristics:

- **Origin**: The dataset is the result of a chemical analysis of wines grown in the same region in Italy but derived from three different cultivars (types of grape). The analysis determined the quantities of 13 constituents found in each of the three types of wines.
- **Features**: The dataset contains 13 different measurements taken from the analysis of the wines, which serve as the features. These measurements include aspects such as:
    - Alcohol content.
    - Malic acid.
    - Ash.
    - Alcalinity of ash.
    - Magnesium content.
    - Total phenols.
    - Flavanoids.
    - Nonflavanoid phenols.
    - Proanthocyanins.
    - Color intensity.
    - Hue.
    - OD280/OD315 of diluted wines.
    - Proline (a type of amino acid).
- **Target Variable**: The target variable in this dataset is the type of wine, which is a categorical variable indicating one of the three possible cultivars.
- **Dataset Size**: The dataset is not very large, making it manageable for training models without requiring significant computational resources. It includes *178 samples*, which is sufficient for learning but small enough for quick experimentation.

In [3]:
# Load the wine dataset
wine = load_wine()

# Create a DataFrame from the Wine dataset
wine_data = pd.DataFrame(wine.data, columns=wine.feature_names)

# Add a 'classes' column to the DataFrame with the target labels
wine_data['classes'] = wine.target_names[wine.target]

# Prepare the feature matrix (X) and the target vector (y)
X = wine_data.drop('classes', axis=1)
Y = wine_data['classes']

# Perform a train-test split
# This example uses 85% of the data for training and 15% for testing
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=6)

# Data frames constructed for training 
data = pd.concat([X_train, Y_train], axis=1)
data_test = pd.concat([X_test, Y_test.reset_index(drop=True)], axis=1)

In [4]:
# Display pair plot


In [5]:
# Select which variables to study closely
j = 6
k = 9
print(wine.feature_names[j], wine.feature_names[k])

flavanoids color_intensity


In [6]:
# plot scatter plot

In [7]:
# Extract a subset of the data focusing on the chosen dimensions and class
df1 = data[[wine.feature_names[j], wine.feature_names[k], 'classes']]

# ----- Estimating the parameters for LDA -------

# Display Gaussian distributions of each class and boundary lines


In [8]:
# Plot LDA decision boundaries with scatter plot


In [9]:
#Classify and display classification report 

In [10]:
# Calculate confusion matrix

## Write Up

Write a blurb describe the results of your work above. Make sure to address the following items within your writeup:

1. Name the two features you used to perform LDA. Make sure to explain that LDA can happen with all 13 features. Why did we only pick two of them?
2. What are the means of the three classes within the 2D feature space? What is the variance-covariance matrix?
3. Explain how effective the model is at classification.